# Import Library

In [1]:
!pip install kaggle

In [2]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
%matplotlib inline

# Import Data

In [3]:
! mkdir ~/.kaggle
! cp kaggle.json ~/.kaggle/
! chmod 600 ~/.kaggle/kaggle.json
! kaggle competitions download -c 114-ml-hw-2-store-sales-time-series-forecasting

100% 22.0M/22.0M [00:00<00:00, 107MB/s]



In [4]:
!unzip 114-ml-hw-2-store-sales-time-series-forecasting

Archive:  114-ml-hw-2-store-sales-time-series-forecasting.zip
  inflating: holidays_events.csv     
  inflating: oil.csv                 
  inflating: sample_submission.csv   
  inflating: stores.csv              
  inflating: test.csv                
  inflating: train.csv               
  inflating: transactions.csv        


In [5]:
# Import
train = pd.read_csv("../content/train.csv")
train = train[train['date'] >= '2017-01-01']
test = pd.read_csv("../content/test.csv")
stores = pd.read_csv("../content/stores.csv")
sub = pd.read_csv("../content/sample_submission.csv")
transactions = pd.read_csv("../content/transactions.csv")
oil = pd.read_csv("../content/oil.csv")
event = pd.read_csv("../content/holidays_events.csv")

# Data Preprocessing

In [ ]:
oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()
oil['oil_7d_ma'] = oil['dcoilwtico'].rolling(window=7, min_periods=1).mean().ffill().bfill()

event = event.drop_duplicates(subset=['date'], keep='first')

In [ ]:
train_oil = pd.merge(train, oil, on = "date", how = 'left')
train_oil = train_oil.fillna(method = 'pad')

test_oil = pd.merge(test, oil, on = "date", how = 'left')
test_oil = test_oil.fillna(method = 'pad')

/tmp/ipykernel_7850/1595028843.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  train_oil = train_oil.fillna(method = 'pad')
/tmp/ipykernel_7850/1595028843.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  test_oil = test_oil.fillna(method = 'pad')


In [ ]:
train_oil_holiday = pd.merge(train_oil, event, on = "date", how = 'left')
train_oil_holiday = train_oil_holiday.fillna('Empty')

test_oil_holiday = pd.merge(test_oil, event, on = "date", how = 'left')
test_oil_holiday = test_oil_holiday.fillna('Empty')

In [ ]:
train_oil_holiday_transactions = train_oil_holiday.copy()
test_oil_holiday_transactions = test_oil_holiday.copy()

In [ ]:
train_oil_holiday_transactions = pd.merge(train_oil_holiday, stores, on = 'store_nbr', how = 'left')

test_oil_holiday_transactions = pd.merge(test_oil_holiday, stores, on =  'store_nbr', how = 'left')

In [ ]:
def split_year(time):
    return int(time.split('-')[0])

def split_month(time):
    return int(time.split('-')[1])

def split_day(time):
    return int(time.split('-')[2])

def weekend(date):
    weekend = []
    a = pd.to_datetime(date)
    for i in range(len(a)):
        if a.iloc[i].weekday() >= 5 :
            weekend.append(1)
        else:
            weekend.append(0)
    return weekend

def weekday(date):
    weekday = []
    a = pd.to_datetime(date)
    for i in range(len(a)):
        weekday.append(a.iloc[i].weekday())
    return weekday

In [ ]:
train_oil_holiday_transactions['Year'] = train_oil_holiday_transactions['date'].apply(split_year)
train_oil_holiday_transactions['Month'] = train_oil_holiday_transactions['date'].apply(split_month)
train_oil_holiday_transactions['Day'] = train_oil_holiday_transactions['date'].apply(split_day)
train_oil_holiday_transactions['Weekend'] = weekend(train_oil_holiday_transactions['date'])
train_oil_holiday_transactions['Weekday'] = weekday(train_oil_holiday_transactions['date'])

In [ ]:
test_oil_holiday_transactions['Year'] = test_oil_holiday_transactions['date'].apply(split_year)
test_oil_holiday_transactions['Month'] = test_oil_holiday_transactions['date'].apply(split_month)
test_oil_holiday_transactions['Day'] = test_oil_holiday_transactions['date'].apply(split_day)
test_oil_holiday_transactions['Weekend'] = weekend(test_oil_holiday_transactions['date'])
test_oil_holiday_transactions['Weekday'] = weekday(test_oil_holiday_transactions['date'])

In [ ]:
for df in [train_oil_holiday_transactions, test_oil_holiday_transactions]:
    df['is_payday'] = df['Day'].apply(lambda x: 1 if x == 15 or x >= 30 else 0)
    df['is_newyear'] = df.apply(lambda x: 1 if x['Month'] == 1 and x['Day'] == 1 else 0, axis=1)

drop_cols = ['id', 'date', 'transferred']
train_oil_holiday_transactions = train_oil_holiday_transactions.drop(columns=drop_cols, errors='ignore')
test_oil_holiday_transactions = test_oil_holiday_transactions.drop(columns=drop_cols, errors='ignore')

train_oil_holiday_transactions = train_oil_holiday_transactions.fillna('None')
test_oil_holiday_transactions = test_oil_holiday_transactions.fillna('None')

In [ ]:
train_oil_holiday_transactions.head(5)

,store_nbr,family,sales,onpromotion,dcoilwtico,oil_7d_ma,type_x,locale,locale_name,description,...,state,type_y,cluster,Year,Month,Day,Weekend,Weekday,is_payday,is_newyear
0,41,PREPARED FOODS,0.0,0,Empty,Empty,Holiday,National,Ecuador,Primer dia del ano,...,El Oro,D,4,2017,1,1,1,6,0,1
1,42,BOOKS,0.0,0,Empty,Empty,Holiday,National,Ecuador,Primer dia del ano,...,Azuay,D,2,2017,1,1,1,6,0,1
2,42,BEVERAGES,0.0,0,Empty,Empty,Holiday,National,Ecuador,Primer dia del ano,...,Azuay,D,2,2017,1,1,1,6,0,1
3,42,BEAUTY,0.0,0,Empty,Empty,Holiday,National,Ecuador,Primer dia del ano,...,Azuay,D,2,2017,1,1,1,6,0,1
4,42,BABY CARE,0.0,0,Empty,Empty,Holiday,National,Ecuador,Primer dia del ano,...,Azuay,D,2,2017,1,1,1,6,0,1


In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

cat_features = ['family', 'type_x', 'locale', 'description', 'locale_name', 'city', 'state', 'type_y']

for col in cat_features:
    encoder = LabelEncoder()

    combined_data = pd.concat([train_oil_holiday_transactions[col], test_oil_holiday_transactions[col]], axis=0).astype(str)
    encoder.fit(combined_data)

    train_oil_holiday_transactions[col] = encoder.transform(train_oil_holiday_transactions[col].astype(str))
    test_oil_holiday_transactions[col] = encoder.transform(test_oil_holiday_transactions[col].astype(str))

In [ ]:
data = train_oil_holiday_transactions.drop(columns = 'sales')
target = train_oil_holiday_transactions['sales']

In [ ]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(data, target, test_size = 0.2, random_state = 5)

# Time Series Model

In [ ]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor

data_full = train_oil_holiday_transactions.drop(columns=['sales'], errors='ignore')
target_full = train_oil_holiday_transactions['sales']

data_full = data_full.apply(pd.to_numeric, errors='coerce')

y_train_log_full = np.log1p(target_full)

model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=12,
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_weight=5,
    random_state=5,
    tree_method='hist',
    n_jobs=-1
)

model.fit(data_full, y_train_log_full)

NameError: name 'train_oil_holiday_transactions' is not defined

# Evaluation

# Submission

In [ ]:
def relu(x):
    relu = []
    for i in x:
        if i < 0:
            relu.append(0)
        else:
            relu.append(i)
    return relu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import numpy as np
from google.colab import files

X_test_final = test_oil_holiday_transactions.drop(columns=['sales'], errors='ignore')

pdt_log = model.predict(X_test_final)

pdt = np.expm1(pdt_log)

sub['sales'] = [0 if i < 0 else i for i in pdt]

output_dir = '/content/drive/MyDrive/Colab Notebooks/hw2'
os.makedirs(output_dir, exist_ok=True)
file_path = os.path.join(output_dir, 'baseline.csv')

sub.to_csv(file_path, index=False)
files.download(file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from google.colab import files

train = pd.read_csv("train.csv")
train = train[train['date'] >= '2017-01-01']
test = pd.read_csv("test.csv")
stores = pd.read_csv("stores.csv")
oil = pd.read_csv("oil.csv")
event = pd.read_csv("holidays_events.csv")
sub = pd.read_csv("sample_submission.csv")

oil['dcoilwtico'] = oil['dcoilwtico'].ffill().bfill()
oil['oil_7d_ma'] = oil['dcoilwtico'].rolling(window=7, min_periods=1).mean().ffill().bfill()

event = event[event['transferred'] == False]
event = event.drop_duplicates(subset=['date'], keep='first')

train_df = train.merge(oil, on='date', how='left')
train_df = train_df.merge(stores, on='store_nbr', how='left')
train_df = train_df.merge(event, on='date', how='left')

test_df = test.merge(oil, on='date', how='left')
test_df = test_df.merge(stores, on='store_nbr', how='left')
test_df = test_df.merge(event, on='date', how='left')

for df in [train_df, test_df]:
    df['date'] = pd.to_datetime(df['date'])
    df['Year'] = df['date'].dt.year
    df['Month'] = df['date'].dt.month
    df['Day'] = df['date'].dt.day
    df['Weekday'] = df['date'].dt.weekday
    df['DayOfYear'] = df['date'].dt.dayofyear
    df['is_payday'] = df['Day'].apply(lambda x: 1 if x == 15 or x >= 30 else 0)
    df['is_weekend'] = df['Weekday'].apply(lambda x: 1 if x >= 5 else 0)

    fill_cols = ['type_y', 'locale', 'locale_name', 'description', 'transferred', 'type_x']
    for c in fill_cols:
        if c in df.columns:
            df[c] = df[c].fillna('None')

cat_features = ['family', 'city', 'state', 'type_x', 'type_y', 'locale', 'locale_name', 'description', 'transferred']
for col in cat_features:
    le = LabelEncoder()
    combined = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
    le.fit(combined)
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col] = le.transform(test_df[col].astype(str))

drop_cols = ['id', 'date']
X_train = train_df.drop(columns=drop_cols + ['sales'], errors='ignore')
y_train = train_df['sales']
X_test = test_df.drop(columns=drop_cols + ['sales'], errors='ignore')

X_train = X_train.apply(pd.to_numeric, errors='coerce')
X_test = X_test.apply(pd.to_numeric, errors='coerce')
y_train_log = np.log1p(y_train)

model = XGBRegressor(
    n_estimators=2500,
    learning_rate=0.016,
    max_depth=11,
    subsample=0.85,
    colsample_bytree=0.8,
    min_child_weight=6,
    random_state=42,
    tree_method='hist',
    n_jobs=-1
)
model.fit(X_train, y_train_log)

pdt_log = model.predict(X_test)
pdt = np.expm1(pdt_log)
pdt[pdt < 0] = 0

sub['sales'] = pdt
sub.to_csv('baseline.csv', index=False)
files.download('baseline.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>